# Stacking ensemble

Combines the base-model predictions from `../predictions/` with a second-stage regressor and writes the final scores to `../results/`. No GPU is needed; this notebook only reads the saved prediction files.

**How the meta-regressor is trained.** The base models are fine-tuned on the train split and their dev-split predictions are used as the meta-training set. Each meta-regressor is fitted with 5-fold cross-validation over the dev rows: every dev prediction reported below is out-of-fold (the row was not in that fold's training data), and the test prediction is the average of the five fold models. Mean blending is included as a non-learned baseline.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import (ARDRegression, BayesianRidge, LinearRegression,
                                  PassiveAggressiveRegressor, TheilSenRegressor)
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

# PassiveAggressiveRegressor is deprecated in recent scikit-learn but still works
warnings.filterwarnings("ignore", category=FutureWarning)

PRED_DIR = "../predictions"
OUT_DIR = "../results"
SEED = 42
N_SPLITS = 5

# base models fed to the meta-regressor (file prefixes in ../predictions)
STACK_MODELS = ["deberta", "structbert", "sentencebert", "roberta"]

## Load the base-model predictions

Rows are sorted by `id` so that every model's predictions line up.

In [2]:
def load_split(split, models=STACK_MODELS):
    columns, labels, ids = [], None, None
    for name in models:
        df = pd.read_csv(f"{PRED_DIR}/{name}_{split}.csv").sort_values("id").reset_index(drop=True)
        columns.append(df["prediction"].rename(name))
        labels, ids = df["label"].values, df["id"].values
    return pd.concat(columns, axis=1).values, labels, ids


X_dev, y_dev, dev_ids = load_split("dev")
X_test, y_test, test_ids = load_split("test")
print("dev :", X_dev.shape, "| test:", X_test.shape)

dev : (2419, 4) | test: (3024, 4)


## Meta-regressors

In [3]:
def rmse_mae(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred)), mean_absolute_error(y_true, y_pred)


def meta_regressors():
    return {
        "DecisionTree": DecisionTreeRegressor(max_depth=5, random_state=SEED),
        "PassiveAggressive": PassiveAggressiveRegressor(max_iter=1000, random_state=SEED),
        "LinearRegression": LinearRegression(),
        "SVR": SVR(kernel="rbf", C=1.0, epsilon=0.1),
        "TheilSen": TheilSenRegressor(random_state=SEED),
        "BayesianRidge": BayesianRidge(),
        "ARD": ARDRegression(),
    }


def stack_with_cv(X_dev, y_dev, X_test, y_test):
    kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    predictions, rows = {}, []

    for name, model in meta_regressors().items():
        dev_oof = np.zeros(len(X_dev))
        fold_test_preds = []
        for train_idx, val_idx in kfold.split(X_dev):
            model.fit(X_dev[train_idx], y_dev[train_idx])
            dev_oof[val_idx] = model.predict(X_dev[val_idx])
            fold_test_preds.append(model.predict(X_test))
        predictions[name] = (dev_oof, np.mean(fold_test_preds, axis=0))

    # baseline: plain average of the base models, nothing to fit
    predictions["MeanBlend"] = (X_dev.mean(axis=1), X_test.mean(axis=1))

    for name, (dev_pred, test_pred) in predictions.items():
        dev_rmse, dev_mae = rmse_mae(y_dev, dev_pred)
        test_rmse, test_mae = rmse_mae(y_test, test_pred)
        rows.append({"Model": name, "Dev_RMSE": dev_rmse, "Test_RMSE": test_rmse,
                     "Dev_MAE": dev_mae, "Test_MAE": test_mae})

    return predictions, pd.DataFrame(rows)

In [4]:
predictions, summary = stack_with_cv(X_dev, y_dev, X_test, y_test)
summary.round(4)

,Model,Dev_RMSE,Test_RMSE,Dev_MAE,Test_MAE
0,DecisionTree,0.5212,0.4923,0.4146,0.3934
1,PassiveAggressive,0.5249,0.5029,0.4270,0.4105
2,LinearRegression,0.4907,0.4858,0.3941,0.3890
3,SVR,0.4995,0.4903,0.3987,0.3901
4,TheilSen,0.4919,0.4872,0.3931,0.3886
5,BayesianRidge,0.4907,0.4858,0.3942,0.3891
6,ARD,0.4908,0.4858,0.3943,0.3891
7,MeanBlend,0.5129,0.5060,0.4187,0.4111


## Save

The summary table and each meta-regressor's dev and test predictions are written to `../results/`.

In [5]:
os.makedirs(f"{OUT_DIR}/stacked_predictions", exist_ok=True)

for name, (dev_pred, test_pred) in predictions.items():
    pd.DataFrame({"id": dev_ids, "prediction": dev_pred}).to_csv(
        f"{OUT_DIR}/stacked_predictions/{name}_dev.csv", index=False)
    pd.DataFrame({"id": test_ids, "prediction": test_pred}).to_csv(
        f"{OUT_DIR}/stacked_predictions/{name}_test.csv", index=False)

summary.to_csv(f"{OUT_DIR}/stacking_summary.csv", index=False)